# Log Anomaly Detection - Temporal Sequence Models
## Transformers, LSTM, GRU, and RNN with Timestamp Features

This notebook implements temporal models that incorporate timestamps and log sequences.

**Models Implemented:**
- Vanilla RNN (baseline sequential)
- LSTM (Long Short-Term Memory)
- GRU (Gated Recurrent Unit)
- Transformer (with positional encoding)

**Key Innovation:** Time-aware features with temporal delta encoding

**Evaluation:** Classification metrics + temporal sequence analysis


In [2]:
!pip3 install pandas numpy matplotlib seaborn scikit-learn torch 

  Using cached pandas-2.3.3-cp311-cp311-macosx_10_9_x86_64.whl (11.6 MB)
  Using cached numpy-2.3.4-cp311-cp311-macosx_14_0_x86_64.whl (6.9 MB)
  Using cached matplotlib-3.10.7-cp311-cp311-macosx_10_12_x86_64.whl (8.3 MB)
  Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
  Using cached scikit_learn-1.7.2-cp311-cp311-macosx_10_9_x86_64.whl (9.3 MB)
  Using cached torch-2.2.2-cp311-none-macosx_10_9_x86_64.whl (150.8 MB)
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_10_9_x86_64.whl (288 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached fonttools-4.60.1-cp311-cp311-macosx_10_9_x86_64.whl (2.4 MB)
  Using cached kiwisolver-1.4.9-cp311-cp311-macosx_10_9_x86_64.whl (66 kB)
  Using cached pillow-12.0.0-cp311-cp311-macosx_10_10_x86_64.whl (5.3 MB)
  Using cached pyparsing-3.2.5-py3-none-any.whl (113 kB)
  Using cached scipy-1.16.3-cp311-cp311-maco

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os, re, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, roc_curve

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

print('✓ All packages imported')



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/michalklos/Documents/Studia2025/Magisterka/Notebooks/.venv-temporal/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/michalklos/Documents/Studia2025/Magisterka/Notebooks/.venv-temporal/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/michalklos/Documents/Studia20

✓ All packages imported


In [ ]:
# Create temporal log sequences with timestamps

def create_temporal_data(n_seq=1000, seq_len=15):
    normal_logs = [
        'Receiving block blk_{} from node {}',
        'Received block blk_{} of size {} from {}',
        'Verification succeeded for blk_{}',
        'INFO DataNode: Processing block blk_{}'
    ]

    anomaly_logs = [
        'Exception in receiveBlock for block blk_{} IOException',
        'WARN DataNode: Disk error on block blk_{}',
        'ERROR: Failed to transfer block blk_{}',
        'Exception verification blk_{} IOException'
    ]

    data = []
    for label in [0, 1]:
        n = 800 if label == 0 else 200
        for _ in range(n):
            seq = []
            ts = []
            current_time = 0

            for step in range(seq_len):
                delta = np.random.randint(100, 5000)
                current_time += delta
                ts.append(current_time / 1000)

                if label == 0:
                    template = np.random.choice(normal_logs)
                elif step < seq_len - 3:
                    template = np.random.choice(normal_logs)
                else:
                    template = np.random.choice(anomaly_logs)

                log_msg = template.format(*[np.random.randint(10, 999) for _ in range(template.count('{}'))])
                seq.append(log_msg)

            data.append({'seq': ' ||| '.join(seq), 'ts': ts, 'label': label})

    return pd.DataFrame(data)

df = create_temporal_data(n_seq=1000, seq_len=15)
print(f'Dataset: {len(df)} sequences | Normal: {len(df[df.label==0])} | Anomaly: {len(df[df.label==1])}')


In [ ]:
# Parse logs and extract features

class LogParser:
    def __init__(self):
        self.templates = {}
        self.tid = 0

    def parse(self, log):
        log = re.sub(r'\d+', '<NUM>', log)
        return log.strip()

    def to_id(self, template):
        if template not in self.templates:
            self.templates[template] = self.tid
            self.tid += 1
        return self.templates[template]

parser = LogParser()

X_ids = []
X_deltas = []
y = df['label'].values

for idx, row in df.iterrows():
    logs = row['seq'].split(' ||| ')
    ts = row['ts']

    ids = [parser.to_id(parser.parse(log)) for log in logs]
    deltas = [0] + [ts[i] - ts[i-1] for i in range(1, len(ts))]

    X_ids.append(np.array(ids, dtype=np.int32))
    X_deltas.append(np.array(deltas, dtype=np.float32))

X_ids = np.array(X_ids, dtype=object)
X_deltas = np.array(X_deltas, dtype=object)

X_id_train, X_id_test, X_delta_train, X_delta_test, y_train, y_test = train_test_split(
    X_ids, X_deltas, y, test_size=0.2, random_state=42, stratify=y
)

vocab_size = len(parser.templates) + 1
print(f'Vocab: {vocab_size} | Train: {len(X_id_train)} | Test: {len(X_id_test)}')


In [ ]:
# PyTorch Dataset

class TemporalDataset(Dataset):
    def __init__(self, ids, deltas, labels):
        self.ids = ids
        self.deltas = deltas
        self.labels = labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return {
            'ids': torch.LongTensor(self.ids[idx]),
            'deltas': torch.FloatTensor(self.deltas[idx]),
            'label': torch.LongTensor([self.labels[idx]]),
            'length': len(self.ids[idx])
        }

def collate_fn(batch):
    ids = nn.utils.rnn.pad_sequence([b['ids'] for b in batch], batch_first=True)
    deltas = nn.utils.rnn.pad_sequence([b['deltas'].unsqueeze(-1) for b in batch], batch_first=True).squeeze(-1)
    labels = torch.cat([b['label'] for b in batch])
    lengths = torch.LongTensor([b['length'] for b in batch])
    return ids, deltas, labels, lengths

train_dataset = TemporalDataset(X_id_train, X_delta_train, y_train)
test_dataset = TemporalDataset(X_id_test, X_delta_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

print('✓ Data loaders created')


In [ ]:
print('\n' + '='*60)
print('TEMPORAL MODELS: Architectures')
print('='*60)

# 1. Vanilla RNN
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hid_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.RNN(emb_dim + 1, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, 2)

    def forward(self, ids, deltas, lengths):
        emb = self.embedding(ids)
        x = torch.cat([emb, deltas.unsqueeze(-1)], dim=-1)
        _, hidden = self.rnn(x)
        return self.fc(hidden.squeeze(0))

# 2. LSTM
class LSTM_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hid_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim + 1, hid_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hid_dim, 2)

    def forward(self, ids, deltas, lengths):
        emb = self.embedding(ids)
        x = torch.cat([emb, deltas.unsqueeze(-1)], dim=-1)
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

# 3. GRU
class GRU_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hid_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim + 1, hid_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hid_dim, 2)

    def forward(self, ids, deltas, lengths):
        emb = self.embedding(ids)
        x = torch.cat([emb, deltas.unsqueeze(-1)], dim=-1)
        _, hidden = self.gru(x)
        return self.fc(hidden[-1])

# 4. Transformer
class Transformer_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hid_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.time_fc = nn.Linear(1, emb_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim * 2, nhead=4, dim_feedforward=hid_dim,
            dropout=0.2, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(emb_dim * 2, 2)

    def forward(self, ids, deltas, lengths):
        emb = self.embedding(ids)
        t_emb = self.time_fc(deltas.unsqueeze(-1))
        x = torch.cat([emb, t_emb], dim=-1)
        mask = self._get_mask(lengths, x.device)
        output = self.transformer(x, src_key_padding_mask=mask)
        pool = output.mean(dim=1)
        return self.fc(pool)

    def _get_mask(self, lengths, device):
        max_len = lengths.max().item()
        return torch.arange(max_len, device=device).unsqueeze(0) >= lengths.unsqueeze(1)

models = {
    'RNN': VanillaRNN(vocab_size),
    'LSTM': LSTM_Model(vocab_size),
    'GRU': GRU_Model(vocab_size),
    'Transformer': Transformer_Model(vocab_size)
}

print('✓ All 4 models defined')


In [ ]:
def train_model(model, train_loader, test_loader, epochs=30, name='Model'):
    print(f'\nTraining {name}...')
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    val_accs = []

    for epoch in range(epochs):
        model.train()
        loss = 0
        for ids, deltas, labels, lengths in train_loader:
            optimizer.zero_grad()
            logits = model(ids, deltas, lengths)
            l = criterion(logits, labels.squeeze())
            l.backward()
            optimizer.step()
            loss += l.item()

        train_losses.append(loss / len(train_loader))

        model.eval()
        with torch.no_grad():
            correct = 0
            for ids, deltas, labels, lengths in test_loader:
                logits = model(ids, deltas, lengths)
                pred = logits.argmax(dim=1)
                correct += (pred == labels.squeeze()).sum().item()

            val_acc = correct / len(test_dataset)
            val_accs.append(val_acc)

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/30 | Loss: {train_losses[-1]:.6f} | Val Acc: {val_acc:.4f}')

    return train_losses, val_accs

# Train all models
results = {}
for name, model in models.items():
    losses, accs = train_model(model, train_loader, test_loader, name=name)
    results[name] = {'losses': losses, 'accs': accs}

# Plot training
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name in models.keys():
    ax[0].plot(results[name]['losses'], label=name)
    ax[1].plot(results[name]['accs'], label=name)

ax[0].set_title('Training Loss', fontweight='bold')
ax[0].set_ylabel('Loss')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].set_title('Validation Accuracy', fontweight='bold')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate models

print('\n' + '='*60)
print('MODEL EVALUATION')
print('='*60)

eval_results = {}

for name, model in models.items():
    print(f'\n{name}:')
    model.eval()

    preds_all = []
    labels_all = []
    probs_all = []

    with torch.no_grad():
        for ids, deltas, labels, lengths in test_loader:
            logits = model(ids, deltas, lengths)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.squeeze().cpu().numpy())
            probs_all.extend(probs[:, 1].cpu().numpy())

    preds_all = np.array(preds_all)
    labels_all = np.array(labels_all)
    probs_all = np.array(probs_all)

    acc = accuracy_score(labels_all, preds_all)
    prec, rec, f1, _ = precision_recall_fscore_support(labels_all, preds_all, average='binary')
    auc = roc_auc_score(labels_all, probs_all)

    print(f'  Accuracy: {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall: {rec:.4f}')
    print(f'  F1-Score: {f1:.4f}')
    print(f'  ROC-AUC: {auc:.4f}')

    eval_results[name] = {
        'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'auc': auc,
        'preds': preds_all, 'labels': labels_all, 'probs': probs_all
    }


In [ ]:
# Comparison table

results_table = pd.DataFrame({
    'Model': list(eval_results.keys()),
    'Accuracy': [eval_results[m]['acc'] for m in eval_results],
    'Precision': [eval_results[m]['prec'] for m in eval_results],
    'Recall': [eval_results[m]['rec'] for m in eval_results],
    'F1': [eval_results[m]['f1'] for m in eval_results],
    'ROC-AUC': [eval_results[m]['auc'] for m in eval_results]
})

print('\n' + '='*80)
print('RESULTS SUMMARY')
print('='*80)
print(results_table.to_string(index=False))

# Visualize
fig, ax = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['acc', 'prec', 'rec', 'f1']
titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax_pos = ax[idx // 2, idx % 2]
    values = [eval_results[m][metric] for m in eval_results]
    ax_pos.bar(eval_results.keys(), values, color=['skyblue', 'lightcoral', 'lightgreen', 'plum'])
    ax_pos.set_title(title, fontweight='bold')
    ax_pos.set_ylabel('Score')
    ax_pos.set_ylim([0, 1])
    ax_pos.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# ROC Curves

fig, ax = plt.subplots(figsize=(10, 8))

for name in eval_results.keys():
    fpr, tpr, _ = roc_curve(eval_results[name]['labels'], eval_results[name]['probs'])
    auc = eval_results[name]['auc']
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curves - Temporal Models', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Results Summary & Recommendations

### Performance Comparison

| Model | Pros | Cons | Best For |
|-------|------|------|----------|
| **RNN** | Simple, fast | Vanishing gradients | Baseline |
| **LSTM** | Memory cells, long dependencies | More parameters, slower | Complex temporal patterns |
| **GRU** | Efficient, comparable to LSTM | Simplified architecture | Resource-constrained |
| **Transformer** | Parallel processing, attention | Requires GPU, more memory | Long sequences, production |

### Timestamp Integration Methods

1. **Direct Concatenation** (RNN/LSTM/GRU)
   - Time delta concatenated with embeddings
   - Effective for short-to-medium sequences

2. **Positional Encoding** (Transformer)
   - Learnable position embeddings
   - Captures relative timing relationships

### Next Steps

1. **Phase 3: Advanced Anomaly Detection**
   - Ensemble multiple models
   - Reconstruction-based scoring
   - Threshold optimization

2. **Production Deployment**
   - Model selection based on latency/accuracy trade-off
   - Online learning for concept drift
   - Temporal regularization

3. **Further Optimization**
   - Time-aware LSTM variants
   - Attention visualization
   - Multi-task learning
